# AETH Parquet in Zoer JupyterLab

## Goal
Verify that the separately hosted JupyterLab can read the same columnar AETH extract as the Zoer Data Space. This is a data access and query check, not a scientific analysis.

## Setup
The Zoer-hosted JupyterLab mounts the hosted Datasets collection read-only. Run this notebook there without uploading a second copy of the Parquet file. `zoer_datasets.list_datasets()` shows what is currently available, and `source_path()` resolves the AETH Parquet. The local fallback uses the versioned recipe output when running from this repository.

The input comes from `unified_filter_parquet.manifest.json` version 1. Expected counts were checked against Zoer dataset `caa04b1f-7f86-4c34-b059-a5a8d04c7501` on 2026-09-22.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import pyarrow.parquet as pq

DATASET_ID = "caa04b1f-7f86-4c34-b059-a5a8d04c7501"
try:
    from zoer_datasets import list_datasets, source_path
except ModuleNotFoundError:
    candidates = [
        Path("unified_filter_dataset.parquet"),
        Path("../../output/tables/unified_filter_dataset.parquet"),
    ]
    parquet_path = next((candidate for candidate in candidates if candidate.is_file()), None)
    if parquet_path is None:
        raise FileNotFoundError("Run the AETH Parquet recipe locally or use the Zoer-hosted JupyterLab")
    print("Local recipe output")
else:
    available = pd.DataFrame(list_datasets())
    display(available[["id", "name", "status", "rows"]])
    parquet_path = source_path(DATASET_ID, "unified_filter_dataset.parquet")

parquet_path = parquet_path.resolve()
print(f"Reading {parquet_path.name} ({parquet_path.stat().st_size:,} bytes)")

Local recipe output
Reading unified_filter_dataset.parquet (427,578 bytes)


## Steps
### 1. Read metadata and one column
Parquet lets the reader select only the column needed for this check.

In [2]:
parquet_file = pq.ParquetFile(parquet_path)
site = pd.read_parquet(parquet_path, columns=["Site"])["Site"]
print(f"Rows: {parquet_file.metadata.num_rows:,}; row groups: {parquet_file.metadata.num_row_groups}; columns: {len(parquet_file.schema.names)}")
site.value_counts().rename_axis("Site").reset_index(name="rows")

Rows: 44,493; row groups: 1; columns: 24


,Site,rows
0,CHTS,16803
1,USPA,13216
2,ETAD,11374
3,INDH,3100


### 2. Query the same file with DuckDB
The SQL computes aggregates directly from Parquet without loading every column into a pandas DataFrame.

In [3]:
counts = duckdb.sql("""
    SELECT COUNT(*) AS row_count,
           COUNT(*) FILTER (WHERE Site = 'ETAD') AS etad_count,
           COUNT(DISTINCT Site) AS site_count
    FROM read_parquet(?)
""", params=[str(parquet_path)]).df()
counts

,row_count,etad_count,site_count
0,44493,11374,4


## Checks
Compare independent reads and the version 1 baseline. If these fail, confirm that you uploaded the exact Parquet export registered in the Data Space.

In [4]:
observed = {key: int(counts.iloc[0][key]) for key in ("row_count", "etad_count", "site_count")}
from_pandas = {"row_count": len(site), "etad_count": int(site.eq("ETAD").sum()), "site_count": int(site.nunique())}
expected = {"row_count": 44_493, "etad_count": 11_374, "site_count": 4}
assert observed == from_pandas, (observed, from_pandas)
assert observed == expected, (observed, expected)
print(f"Verified: {observed}")

Verified: {'row_count': 44493, 'etad_count': 11374, 'site_count': 4}


## Next Steps
Use DuckDB projections and filters for larger queries, and keep the Parquet source tied to the versioned manifest. Scientific exclusions and filtering still belong in the existing AETH analysis helpers; this notebook does not apply them. The Zoer Data Space remains the hosted source of truth.